In [2]:
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FunctionTransformer
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split


import re

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sqlalchemy.testing.plugin.plugin_base import stop_test_class
from toolz import tail



In [3]:
df = pd.read_csv('IMDB Dataset.csv')
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [4]:
X = df['review']
y = df['sentiment']
X

0        One of the other reviewers has mentioned that ...
1        A wonderful little production. <br /><br />The...
2        I thought this was a wonderful way to spend ti...
3        Basically there's a family where a little boy ...
4        Petter Mattei's "Love in the Time of Money" is...
                               ...                        
49995    I thought this movie did a down right good job...
49996    Bad plot, bad dialogue, bad acting, idiotic di...
49997    I am a Catholic taught in parochial elementary...
49998    I'm going to have to disagree with the previou...
49999    No one expects the Star Trek movies to be high...
Name: review, Length: 50000, dtype: object

In [5]:
X_train,X_test, y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,shuffle=True)
print(len(y_train))
X_train.shape


40000


(40000,)

In [6]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')
my_tokenizer = word_tokenize

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\top\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\top\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [7]:
nltk.download('stopwords')
my_stops=stopwords.words('english')
my_stops

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\top\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [8]:
my_tokenizer = word_tokenize

In [9]:
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')
my_stops=stopwords.words('english')
tokenizer  = word_tokenize

def preprocess(texts):
    if not isinstance(texts,str):
        return " "

    text = texts.lower()
    text = re.sub(r'\d',' ',text)
    text = re.sub(r'[^\w\s]',' ',text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'(.)\1{2,}',' ',text)

    text = [word for word in tokenizer(text) if word not in my_stops ]

    return  " ".join(text)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\top\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\top\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [10]:
text = ' "My" "name" is Hossein Moein'
preprocess(text)

'name hossein moein'

In [11]:
print(X_train)
print(X_test)

39087    That's what I kept asking myself during the ma...
30893    I did not watch the entire movie. I could not ...
45278    A touching love story reminiscent of In the M...
16398    This latter-day Fulci schlocker is a totally a...
13653    First of all, I firmly believe that Norwegian ...
                               ...                        
11284    `Shadow Magic' recaptures the joy and amazemen...
44732    I found this movie to be quite enjoyable and f...
38158    Avoid this one! It is a terrible movie. So wha...
860      This production was quite a surprise for me. I...
15795    This is a decent movie. Although little bit sh...
Name: review, Length: 40000, dtype: object
33553    I really liked this Summerslam due to the look...
9427     Not many television shows appeal to quite as m...
199      The film quickly gets to a major chase scene w...
12447    Jane Austen would definitely approve of this o...
39489    Expectations were somewhat high for me when I ...
             

In [12]:
X_train = X_train.apply(preprocess)
X_test = X_test.apply(preprocess)

In [13]:
X_train

39087    kept asking many fights screaming matches swea...
30893    watch entire movie could watch entire movie st...
45278    touching love story reminiscent mood love draw...
16398    latter day fulci schlocker totally abysmal con...
13653    first firmly believe norwegian movies continua...
                               ...                        
11284    shadow magic recaptures joy amazement first mo...
44732    found movie quite enjoyable fairly entertainin...
38158    avoid one terrible movie exciting pointless mu...
860      production quite surprise absolutely love obsc...
15795    decent movie although little bit short time pa...
Name: review, Length: 40000, dtype: object

In [14]:
tfid = TfidfVectorizer(
    min_df=5,
    max_features=25000,
    max_df=.9
)


In [15]:
classifier = LinearSVC(C=.5)

In [16]:
base_pipeline = Pipeline([
    ('tfidf', tfid),
    ('classifier', classifier)
])

In [17]:
stk = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

In [19]:
grid_model = GridSearchCV(
    estimator=base_pipeline,
    param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [2, 3],
    "tfidf__max_df": [0.85, 0.95],
    "classifier__C": [0.5, 1]
},
    scoring='f1_macro',
    n_jobs=-1,
    cv= stk,
    verbose=3

)

grid_model.fit(X_train,y_train)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


,estimator,Pipeline(step...rSVC(C=0.5))])
,param_grid,"{'classifier__C': [0.5, 1], 'tfidf__max_df': [0.85, 0.95], 'tfidf__min_df': [2, 3], 'tfidf__ngram_range': [(1, ...), (1, ...)]}"
,scoring,'f1_macro'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,input,'content'


In [20]:
grid_model.best_params_

{'classifier__C': 0.5,
 'tfidf__max_df': 0.85,
 'tfidf__min_df': 2,
 'tfidf__ngram_range': (1, 2)}

In [21]:
grid_model.best_score_

0.9011695093110681

In [23]:
best_model = grid_model.best_estimator_

In [24]:
y_pred = best_model.predict(X_test)

In [25]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

    negative       0.91      0.90      0.90      4961
    positive       0.90      0.91      0.91      5039

    accuracy                           0.91     10000
   macro avg       0.91      0.91      0.91     10000
weighted avg       0.91      0.91      0.91     10000



In [39]:
feature_names = best_model.named_steps['tfidf'].get_feature_names_out()
feature_names

weights = best_model.named_steps['classifier'].coef_[0]

0.3698749843966703


In [41]:
importance = pd.DataFrame({
    'features':feature_names,
'weights': weights
})
importance

,features,weights
0,aamir,0.369875
1,aardman,0.109950
2,aaron,0.313709
3,ab,0.267555
4,abandon,-0.010773
...,...,...
24995,zooms,0.115712
24996,zorro,0.284150
24997,zu,0.463135
24998,zucco,-0.585567


In [43]:
importance.sort_values(by='weights', ascending=False, inplace=True)
importance

,features,weights
7154,excellent,2.984672
9692,great,2.846670
16573,perfect,2.491370
3006,brilliant,2.426236
24209,well worth,2.311272
...,...,...
5839,disappointment,-2.917726
2391,boring,-3.451832
23913,waste,-3.519589
1474,awful,-3.672804


In [44]:
importance.head(50)

,features,weights
7154,excellent,2.984672
9692,great,2.846670
16573,perfect,2.491370
3006,brilliant,2.426236
24209,well worth,2.311272
782,amazing,2.245263
24496,wonderful,2.211237
16582,perfectly,2.149031
10385,hilarious,2.142467
6691,enjoyable,2.139417


In [46]:
importance.tail(50)

,features,weights
15661,oh,-1.584985
21953,tedious,-1.601845
1636,badly,-1.606505
15196,neither,-1.647403
21439,stupid,-1.668291
23292,uninteresting,-1.685516
6123,dreadful,-1.686862
16387,pathetic,-1.709530
13994,mildly,-1.733794
12994,lousy,-1.738929


In [50]:
# get new text to find label
new_comment = input("Enter Comment : ")
best_model.predict([new_comment])

array(['positive'], dtype=object)

In [51]:
# Pickle - works fine
#import pickle
#with open('model.pkl', 'wb') as f:
 #   pickle.dump(best_model, f)

#with open('model.pkl', 'rb') as f:
 #   model = pickle.load(f)

# Joblib - recommended for sklearn
import joblib
joblib.dump(best_model, 'model.joblib')
model = joblib.load('model.joblib')